# Stage 0: Prompt Sensitivity & Selection (`run_prompt_sensitivity.py`)
Evaluates zero-shot sentiment classification performance (`Accuracy`, `F1`, `ROC-AUC`, `Agreement vs. Base`) across 4 prompt templates on SST-2 `train` (with optional attribution stability check).

### 1. GPU Environment Verification

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB) | bf16: {torch.cuda.is_bf16_supported()}")

### 2. Google Drive Mounting, Repository Setup, and Dependencies

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')
DRIVE_RESULTS = "/content/drive/MyDrive/llm_xai_results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# 2. GitHub Configuration
GITHUB_TOKEN = "" #@param {type:"string"}
REPO_OWNER_REPO = "jakubstachyra/llm-xai" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

try:
    from google.colab import userdata
    token_from_secrets = userdata.get('GH_TOKEN')
    if token_from_secrets:
        GITHUB_TOKEN = token_from_secrets
except Exception:
    pass

if GITHUB_TOKEN.strip():
    CLONE_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_OWNER_REPO}.git"
else:
    CLONE_URL = f"https://github.com/{REPO_OWNER_REPO}.git"

REPO_DIR = "/content/llm-xai"

if not os.path.exists(REPO_DIR):
    res = !git clone -b {BRANCH} {CLONE_URL} {REPO_DIR}
    print("\n".join(res))
    if not os.path.exists(REPO_DIR):
        raise RuntimeError("Cloning error! If repository is private, provide PAT in GITHUB_TOKEN or GH_TOKEN secret.")
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

# 3. Dependencies
!pip install -q --upgrade numpy "transformers>=4.43.0,<4.47.0" "accelerate>=0.30.0" "datasets>=2.19.0" scikit-learn scipy pandas matplotlib huggingface_hub

# 4. HuggingFace Token (for gated models)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

### 3. Run Prompt Sensitivity / Selection (`Qwen/Qwen3-8B-Base` on `train`)

In [ ]:
model = "Qwen/Qwen3-8B-Base" #@param ["Qwen/Qwen3-8B-Base", "Qwen/Qwen3-1.7B-Base", "Qwen/Qwen3-0.6B-Base", "microsoft/phi-2"]
split = "train" #@param ["train", "validation"]
clf_samples = 872 #@param {type:"integer"}
attr_samples = 0 #@param {type:"integer"}
methods = "grad_x_input,occlusion,ig" #@param {type:"string"}
seed = 42 #@param {type:"integer"}
no_calibrate = False #@param {type:"boolean"}
auto_shutdown = True #@param {type:"boolean"}

tag = model.split("/")[-1]
output_dir = f"{DRIVE_RESULTS}/prompt_sensitivity_{tag}_{split}"
os.makedirs(output_dir, exist_ok=True)
cal_flag = "--no-calibrate" if no_calibrate else ""

!python -u scripts/run_prompt_sensitivity.py \
    --model {model} \
    --split {split} \
    --clf-samples {clf_samples} \
    --attr-samples {attr_samples} \
    --methods {methods} \
    --seed {seed} \
    {cal_flag} \
    --output-dir "{output_dir}"

import pandas as pd
from IPython.display import display, Image

clf_csv = os.path.join(output_dir, "prompt_classification_metrics.csv")
if os.path.exists(clf_csv):
    display(pd.read_csv(clf_csv).round(4))

fig_path = os.path.join(output_dir, "prompt_sensitivity_summary.png")
if os.path.exists(fig_path):
    display(Image(filename=fig_path))

if auto_shutdown:
    print("Releasing GPU and stopping Colab runtime...")
    from google.colab import runtime
    runtime.unassign()